# 환경설정

In [ ]:
# 인증


from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

In [ ]:
# 시각화 설정

!pip install koreanize-matplotlib -q
import koreanize_matplotlib
import matplotlib.pyplot as plt

# mart_acquisition — AARRR 1단계 · 획득(Acquisition) Mart

**AARRR 정의 (팀 합의)**
1. **Acquisition(획득)**: 가입 (`stg_users` 기준, 로그성/무효 레코드 및 스탭·관리자 제외) ← 이 노트북
2. Activation(활성화): 첫 수신 + 힌트 열람 경험 + (40명 이상 학교, 4명 이상 반 조건 충족)
3. Retention(유지): 재방문 (기간 추후 논의)
4. 핵심 가설: "첫 수신과 힌트 열람을 모두 경험한 유저는 그렇지 않은 유저에 비해 재방문율이 높을 것"

**목적**: 가입(Acquisition) 시점 기준 유저 1명 = 1행인 base 테이블을 만든다.
이후 Activation/Retention Mart가 이 테이블에 컬럼을 이어 붙이는 방식으로 확장한다
(user_id가 공통 키).

**산출물**
- `mart.mart_acquisition` — 유저 grain (분석·조인용 기준 테이블)
- `mart.mart_acquisition_daily` — 날짜×세그먼트 grain (대시보드 트렌드용 집계)

**적용하는 제외 기준 (이전 EDA에서 확정된 것만 반영, `__Stage.xlsx` 및 클린 버전 노트북 근거)**

| 기준 | 적용 여부 | 근거 |
|---|---|---|
| `is_staff_account = TRUE` (스탭·관리자) | 제외 | AARRR 정의에 명시됨 |
| `is_valid_user = FALSE` (무효/로그성 레코드) | 제외 | AARRR 정의에 명시됨 ("로그 기록" 제외) |
| `ban_status != 'N'` (경고·차단·정지 계열 W/NB/RB) | 제외 | 정상 이용자가 아니므로 획득(Acquisition) 지표에서 제외 |
| `grade NOT IN (1,2,3)` (학년 이상치, 4학년·20학년 등) | 제외 | EDA에서 데이터 오류로 확정 |
| 학교 침투율 100% 초과 | **유지** | PK 중복 없음 확인, 정상 오차로 판단 |
| 동일 device_id 다중 계정 | **유지** | 탐지 신뢰도 낮아 보수적으로 포함 |
| class_num 이상치 | **유지** | 대형 학교의 정상 반편성일 가능성, 원인 미확정 |

⚠️ `is_valid_user`의 정확한 정의(무엇을 기준으로 무효 처리하는지)는 아직 담당자 확인 전입니다.
이번 Mart에서는 "로그 기록 제외"라는 AARRR 정의 문구를 이 플래그로 해석해서 적용했습니다 —
다르게 정의된 거라면 이 노트북의 WHERE 조건만 수정하면 됩니다.


## 0. 필터 적용 전 영향도 확인 (몇 명이 빠지는지 미리 확인)

In [ ]:
%%bigquery df_filter_impact --project your-gcp-project

WITH invalid_groups AS (
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  COUNT(*) AS raw_total,
  COUNTIF(is_staff_account = TRUE) AS staff_excluded,
  COUNTIF(is_valid_user = FALSE) AS invalid_excluded,
  COUNTIF(ban_status != 'N') AS banned_excluded,
  COUNTIF(group_id IN (SELECT group_id FROM invalid_groups)) AS grade_outlier_excluded,
  COUNTIF(
    is_staff_account = FALSE
    AND is_valid_user = TRUE
    AND ban_status = 'N'
    AND (group_id IS NULL OR group_id NOT IN (SELECT group_id FROM invalid_groups))
  ) AS final_remaining
FROM `your-gcp-project.stage.stg_users`;

In [ ]:
df_filter_impact
# final_remaining이 mart_acquisition의 최종 행 수가 됩니다. 네 기준이 겹치는 유저가 있을 수
# 있어 raw_total - (네 excluded 합) 과 정확히 같지 않을 수 있습니다 (정상).

## mart데이터셋 생성

In [ ]:
from google.cloud import bigquery
from google.api_core.exceptions import Conflict

client = bigquery.Client(project="your-gcp-project")

dataset_id = "your-gcp-project.mart"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "asia-northeast3"

try:
    client.create_dataset(dataset)
    print(f"데이터셋 생성 완료: {dataset_id}")
except Conflict:
    print(f"이미 존재함: {dataset_id}")

## 1. mart_acquisition — 유저 grain 기준 테이블

Grain: 1행 = 1명의 유저 (Acquisition 정의를 통과한 유저만)


### 1-1. 최종 정제 미리보기

In [ ]:
%%bigquery df_acquisition_preview --project your-gcp-project

WITH invalid_groups AS (
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  u.user_id,
  DATE(u.signup_at) AS signup_date,
  DATETIME_TRUNC(u.signup_at, SECOND) AS signup_at,
  u.group_id,
  g.school_id,
  s.school_type,
  g.grade,
  g.class_num,
  u.gender,
  s.student_count,
  u.ban_status
FROM `your-gcp-project.stage.stg_users` u
LEFT JOIN `your-gcp-project.stage.stg_group` g
  ON u.group_id = g.group_id
LEFT JOIN `your-gcp-project.stage.stg_school` s
  ON g.school_id = s.school_id
WHERE u.is_staff_account = FALSE
  AND u.is_valid_user = TRUE
  AND u.ban_status = 'N'
  AND (g.group_id IS NULL OR g.group_id NOT IN (SELECT group_id FROM invalid_groups))
ORDER BY u.signup_at DESC
LIMIT 30;

In [ ]:
df_acquisition_preview.head(5)

In [ ]:
df_acquisition_preview.dtypes

In [ ]:
%%bigquery df_hour_check --project your-gcp-project

SELECT
  EXTRACT(HOUR FROM signup_at) AS hour_kst,
  COUNT(*) AS cnt
FROM `your-gcp-project.mart.mart_acquisition`
GROUP BY hour_kst
ORDER BY hour_kst

In [ ]:
df_hour_check

### 1-2. 최종 Mart 생성 (table — Mart는 대시보드가 자주 조회하므로 view가 아닌 table로 생성)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE TABLE `your-gcp-project.mart.mart_acquisition` AS
WITH invalid_groups AS (
  SELECT group_id
  FROM `your-gcp-project.stage.stg_group`
  WHERE grade NOT IN (1, 2, 3)
)
SELECT
  u.user_id,
  DATE(u.signup_at) AS signup_date,
  u.signup_at,
  u.group_id,
  g.school_id,
  s.school_type,
  g.grade,
  g.class_num,
  u.gender,
  s.student_count,
  u.ban_status
FROM `your-gcp-project.stage.stg_users` u
LEFT JOIN `your-gcp-project.stage.stg_group` g
  ON u.group_id = g.group_id
LEFT JOIN `your-gcp-project.stage.stg_school` s
  ON g.school_id = s.school_id
WHERE u.is_staff_account = FALSE
  AND u.is_valid_user = TRUE
  AND u.ban_status = 'N'
  AND (g.group_id IS NULL OR g.group_id NOT IN (SELECT group_id FROM invalid_groups));

### 1-3. 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT user_id) AS distinct_user_id,      -- total_rows와 같아야 함 (user_id가 PK)
  COUNTIF(school_id IS NULL) AS null_school_id,       -- group_id가 없는 유저 수 (반 미배정)
  COUNTIF(ban_status != 'N') AS non_normal_ban_status, -- 0이어야 함 (필터링 확인)
  MIN(signup_date) AS earliest_signup,
  MAX(signup_date) AS latest_signup
FROM `your-gcp-project.mart.mart_acquisition`;

## 2. mart_acquisition_daily — 날짜별 집계 (대시보드 트렌드용)

Grain: 1행 = 1일 × 1 school_type


In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE TABLE `your-gcp-project.mart.mart_acquisition_daily` AS
SELECT
  signup_date,
  school_type,
  COUNT(*) AS signup_count,
  COUNT(DISTINCT school_id) AS active_school_count
FROM `your-gcp-project.mart.mart_acquisition`
GROUP BY signup_date, school_type
ORDER BY signup_date;

### 2-1. 생성 후 검증 + 시각화

In [ ]:
%%bigquery df_daily_check --project your-gcp-project
SELECT * FROM `your-gcp-project.mart.mart_acquisition_daily`
ORDER BY signup_date

In [ ]:
df_daily_check.head(10)

In [ ]:
pivot = df_daily_check.pivot(index='signup_date', columns='school_type', values='signup_count').fillna(0)

plt.figure(figsize=(12, 4))
for col in pivot.columns:
    plt.plot(pivot.index, pivot[col], label=col)
plt.xlabel('가입일')
plt.ylabel('가입자 수')
plt.title('일별 가입자 수 추이 (school_type별)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 다음 단계

- 이 `mart_acquisition`의 `user_id`를 기준으로, Activation Mart(첫 수신 + 힌트 열람 + 학교/반 조건)를
  LEFT JOIN으로 이어 붙이면 됩니다.
- Retention 기간이 아직 미정이니, Activation까지 만들고 나서 재방문 기간을 팀 회의로 확정한 뒤
  Retention Mart로 이어가는 순서를 추천드립니다.
- `is_valid_user`의 정확한 정의가 확인되면, 0번 셀의 `invalid_excluded` 숫자가 맞게 반영된 건지
  다시 한 번 검산해보시는 걸 권장합니다.
